In [1]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import random
from pathlib import Path
import os 
import sys
from collections import Counter

from imblearn.under_sampling import RandomUnderSampler
from sklearn.inspection import permutation_importance

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
BASE_DIR = Path.cwd() 
print(BASE_DIR)

c:\Users\Neezy\Videos\attacks_improved_v2\Attacks-Improved_v2\decision_tree


In [4]:
project_root = os.path.abspath('..')
if project_root not in sys.path: 
    sys.path.append(project_root)

In [5]:
#### log initialization
from utils.logs import logger_module 
logger = logger_module(logger_name=__name__) 

In [6]:
# import attacks
from attack_functionality.general_attack import GeneralAttack 
attack = GeneralAttack()

In [7]:
data = pd.read_csv(BASE_DIR.parent/'data/ds_with_sample_name.csv')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [8]:
X = data.iloc[:,:-1]
y = data.iloc[:,-1]

In [9]:
random_under_sample = RandomUnderSampler(random_state=42)

In [10]:
x_resample, y_resample = random_under_sample.fit_resample(X,y)
balanced_df = pd.DataFrame(x_resample, columns=X.columns)
balanced_df['label'] = y_resample

C:\Users\Neezy\AppData\Local\Temp\ipykernel_28704\380394481.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  balanced_df['label'] = y_resample


In [11]:
balanced_df = balanced_df.copy()

In [12]:
# is_all_zero = (balanced_df.iloc[:, 2:-1].values == 0).all(axis=1)
# counter = 0
# for x in is_all_zero: 
#     if x==True: 
#        counter +=1 
#     else: 
#         print("fine") 

# print(counter) 

In [13]:
balanced_df['label'].value_counts()

label
0    570
1    570
Name: count, dtype: int64

In [14]:
X_bal = balanced_df.iloc[:, 2:-1]
y_bal = balanced_df.iloc[:, -1]
X_train, X_test, y_train, y_test = train_test_split(X_bal, y_bal, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train, y_train)
y_pred = classifier.predict(X_test)

In [15]:
result = permutation_importance(classifier, X_test, y_test, n_repeats=10, scoring='roc_auc', random_state=42)

In [16]:
perm_imp = pd.DataFrame({
    "features":X_bal.columns, 
    "importances_mean": result.importances_mean, 
    "importances std": result.importances_std
}).sort_values('importances_mean', ascending=False)

In [17]:
top_100 = perm_imp.head(20)['features'].to_list()
print(len(top_100))
top_100.append('label')

top_features_df = balanced_df[top_100]

20


In [18]:
top_features_df.head()

,NtQueryAttributesFile,DrawTextExW,RegEnumKeyExW,NtAllocateVirtualMemory,RegOpenKeyExW,LdrLoadDll,NtTerminateProcess,RegEnumValueW,CertControlStore,NtClose,SetErrorMode,Thread32First,Thread32Next,CreateRemoteThread,InternetReadFile,CreateRemoteThreadEx,timeGetTime,DnsQuery_A,InternetGetConnectedState,RegisterHotKey,label
96,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
97,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
99,3,127,192,503,714,49,3,0,0,360,9,0,0,0,0,0,0,0,0,0,0
100,2,169,192,444,709,40,3,0,0,2864,9,0,0,0,0,0,0,0,0,0,0
101,3,294,358,439,890,53,3,0,0,3875,9,0,0,0,0,0,0,0,0,0,0


In [19]:
X_fi = top_features_df.iloc[:,:-1].values
y_fi = top_features_df.iloc[:,-1].values
original_indices = np.arange(len(X_fi)) 
X_train_fi, X_test_fi, y_train_fi, y_test_fi, idx_train, idx_test = train_test_split(X_fi, y_fi, original_indices, test_size=0.2, random_state=42)
classifier = DecisionTreeClassifier(random_state=42)
classifier.fit(X_train_fi, y_train_fi)
y_pred_fi = classifier.predict(X_test_fi)
accuracy_fi = accuracy_score(y_test_fi, y_pred_fi)
print(f'Accuracy: {accuracy_fi * 100:.2f}%')
cm_fi = confusion_matrix(y_test_fi, y_pred_fi)
print(f"Confusion Matrix:\n {cm_fi}") 
cr_fi = classification_report(y_test_fi, y_pred_fi)
print(f"Classification report:\n {cr_fi}")

Accuracy: 94.74%
Confusion Matrix:
 [[125   3]
 [  9  91]]
Classification report:
               precision    recall  f1-score   support

           0       0.93      0.98      0.95       128
           1       0.97      0.91      0.94       100

    accuracy                           0.95       228
   macro avg       0.95      0.94      0.95       228
weighted avg       0.95      0.95      0.95       228



In [20]:
xtrain_candidate_num = round(0.2*len(X_train_fi))
print(xtrain_candidate_num)

182


In [21]:
ransomware_training_sample_indices = np.where(y_train_fi==1)[0]
non_zero_mask = X_train_fi[ransomware_training_sample_indices].sum(axis=1) !=0
valid_indices = ransomware_training_sample_indices[non_zero_mask]

idx = np.random.choice(valid_indices, 182, replace=False)
xtrain_poison_candidates = X_train_fi[idx] 

In [22]:
print(len(xtrain_poison_candidates))

182


In [23]:
perturbation=80
asamples, adv_idx = attack.generate_adversarial_sample(idx, xtrain_poison_candidates, perturbation, classifier)

The average  l2 distance 17.17883646818221 across 174 samples
The biggest  l2 distance 48.0 across 174 samples
The smallest l2 distance 17.0 across 174 samples


In [24]:
print(len(asamples))
print(f"AGR: {len(asamples)/len(xtrain_poison_candidates):.4f}") 

174
AGR: 0.9560


In [25]:
### Adversarial Retrain ###  
mask = np.ones(X_train_fi.shape[0], dtype=bool)
mask[adv_idx] = False

X_train_base_samples_removed = X_train_fi[mask]
y_train_base_samples_removed = y_train_fi[mask]
print(f"length of Xtrain before successful base adversarial samples removed: {len(X_train_fi)}")
print(f"length of Xtrain with successful base adversarial samples removed: {len(X_train_base_samples_removed)}")

### Select a hold out test set of the generated adversarial samples
k = round(0.2 * len(asamples)) 
print(f"20% of the adversarial samples held out for testing: {k}")
remove_samples_from_asamples = random.sample(asamples, k)
print(len(remove_samples_from_asamples))

### remove test set from total adversarial samples and add adversarial samples to the training set 
result = [item for item in asamples if item not in remove_samples_from_asamples]
np_result = np.array(result)
print(len(result))
poisoned_xtrain = np.concatenate((X_train_base_samples_removed, np_result))
print(f"Total length of poisoned Xtrain: {len(poisoned_xtrain)}")

### Add the corresponding amount of ytrain adversarial samples. (correctly labelled as ransomware) 
adv_y = []
for i in range(len(np_result)): 
    adv_y.append(1) 
    
print(len(adv_y))

np_advy = np.array(adv_y)
poisoned_ytrain = np.concatenate((y_train_base_samples_removed, np_advy))
print(f"Total length of poisoned Ytrain: {len(poisoned_ytrain)}") 

if len(poisoned_ytrain) == len(poisoned_xtrain): 
    print("Poisoned Xtrain and ytrain are aligned, posioned training can now commence")
else: 
    print("Not aligned, something is not aligned, check k value")

length of Xtrain before successful base adversarial samples removed: 912
length of Xtrain with successful base adversarial samples removed: 738
20% of the adversarial samples held out for testing: 35
35
120
Total length of poisoned Xtrain: 858
120
Total length of poisoned Ytrain: 858
Poisoned Xtrain and ytrain are aligned, posioned training can now commence


In [26]:
poisoned_classifier = DecisionTreeClassifier(random_state=42)
poisoned_classifier.fit(poisoned_xtrain, poisoned_ytrain)
y_pred_poisoned = poisoned_classifier.predict(X_test_fi)
poisoned_accuracy = accuracy_score(y_test_fi, y_pred_poisoned)
print(f'Accuracy trained on poisoned samples: {poisoned_accuracy * 100:.2f}%')
cm_poisoned = confusion_matrix(y_test_fi, y_pred_poisoned)
print(f"Confusion Matrix trained on poisoned samples:\n {cm_poisoned}") 
cr_poisoned = classification_report(y_test_fi, y_pred_poisoned)
print(f"Classification report trained on poisoned samples:\n {cr_poisoned}")

Accuracy trained on poisoned samples: 95.18%
Confusion Matrix trained on poisoned samples:
 [[126   2]
 [  9  91]]
Classification report trained on poisoned samples:
               precision    recall  f1-score   support

           0       0.93      0.98      0.96       128
           1       0.98      0.91      0.94       100

    accuracy                           0.95       228
   macro avg       0.96      0.95      0.95       228
weighted avg       0.95      0.95      0.95       228



In [27]:
def make_pred_poisoned(samples):
    predictions = [] 
    for sample in samples:
        single_pred = poisoned_classifier.predict([sample])
        predictions.append(single_pred)
        print(single_pred) 
        if single_pred[0] == 1: 
            print("Correctly identified adversarial ransomware sample") 
        else: 
            print("Adversarial sample still adversarial")

    zero_counter = 0 
    one_counter = 0 
    for pred in predictions:
        if pred != 1: 
            zero_counter +=1 
        else:
            one_counter+=1 
        
    
    print(f"Samples still adversarial: {zero_counter}")
    print(f"adversarial_samples identified: {one_counter}")

make_pred_poisoned(remove_samples_from_asamples)

[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correctly identified adversarial ransomware sample
[1]
Correc

In [28]:
###Model transfer ###
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [29]:
### Target Models ####
##  Trained on original data
#   No adversarial samples are added to the training data 

### Random Forest ####
classifier_rf = RandomForestClassifier(n_estimators=20, random_state=42)
classifier_rf.fit(X_train_fi, y_train_fi)
y_pred_rf = classifier_rf.predict(X_test_fi)
accuracy_rf = accuracy_score(y_test_fi, y_pred_rf) 
print(f"Accuracy of Random Forest Target Model: {accuracy_score(y_test_fi, y_pred):.2f}")

### XGboost #### 
params = {
    'objective':'binary:logistic',
    'max_depth':4,
    'learning_rate':0.1,
    'n_estimators':100,
    'alpha':10
}

model = XGBClassifier(**params)
model.fit(X_train_fi, y_train_fi)
y_pred_xgb = model.predict(X_test_fi)
accuracy_xgb = accuracy_score(y_test_fi, y_pred_xgb)
print(f'Accuracy of the XGboost Model: {accuracy_xgb * 100:.2f}%') 

### SVM Classifier Kernel=Poly ###
svm_classifier = SVC(kernel='poly', C=4.0, random_state=42)
svm_classifier.fit(X_train_fi, y_train_fi)
y_pred_svm = svm_classifier.predict(X_test_fi)
print(f"Accuracy of the SVM Classifier (Kernel=Poly): {accuracy_score(y_test_fi, y_pred_svm):.2f}")

### SVM Classifier Kernel=Linear ###
svm_classifier_linear = SVC(kernel='linear', C=4.0, random_state=42)
svm_classifier_linear.fit(X_train_fi, y_train_fi)
y_pred_svm_linear = svm_classifier_linear.predict(X_test_fi)
print(f"Accuracy of the SVM Classifier (Kernel=Linear) : {accuracy_score(y_test_fi, y_pred_svm_linear):.2f}")

### Logistic Regression ###
LR_model = LogisticRegression(solver='saga',max_iter=5000)
LR_model.fit(X_train_fi, y_train_fi)
ypred_LR = LR_model.predict(X_test_fi)
accuracy = accuracy_score(y_test_fi, ypred_LR)
print(f"accuracy of the Logistic Regression Model: {round(accuracy,2)}")

Accuracy of Random Forest Target Model: 0.95
Accuracy of the XGboost Model: 92.11%
Accuracy of the SVM Classifier (Kernel=Poly): 0.44
Accuracy of the SVM Classifier (Kernel=Linear) : 0.93
accuracy of the Logistic Regression Model: 0.81


c:\Users\Neezy\anaconda3\envs\tf\lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [30]:
def evasion_attack(adversarial_samples,
                   RF_model,
                   XGB_Model, 
                   SVM_poly_model, 
                   SVM_linear_model, 
                   Logistic_reg_model):
    
    predictions = {
        "Random_forest":[],
        "XGB":[],
        "SVM_poly": [], 
        "SVM_Linear":[], 
        "Logistic_Regression":[]   
    }
    
    models = {
        "Random_forest": RF_model,
        "XGB":XGB_Model,
        "SVM_poly": SVM_poly_model, 
        "SVM_Linear":SVM_linear_model, 
        "Logistic_Regression":Logistic_reg_model
    }
    
    for name, model in models.items():
        preds = model.predict(adversarial_samples)
        predictions[name] = preds.tolist()
            
    
    return predictions

In [31]:
res = evasion_attack(asamples, classifier_rf, model, svm_classifier, svm_classifier_linear, LR_model)
evaded_ids=[]
evaded_samples = []
original_positions = []
for model_name, preds in res.items():
    print(f"\n {model_name}: predictions")
    total = len(preds)
    count_1 = sum(1 for p in preds if p==1)
    count_2 = total-count_1 
    
    model_evaded_positions = []
    model_evaded_samples = []
    
    for i, p in enumerate(preds): 
        if p == 0: 
           xtrain_position = adv_idx[i] 
           original_position = idx_train[xtrain_position]
           model_evaded_positions.append(original_position)
            
    if model_evaded_positions:
        for og_position in model_evaded_positions:
            og_sample = balanced_df.iloc[og_position]['Sample Name']
            model_evaded_samples.append(og_sample)
        evaded_samples.extend(model_evaded_samples)
        original_positions.extend(model_evaded_positions)
    else: 
        print(f"\n No samples evaded for {model_name}")
    
    print(f"\n {model_name} evasion rate: {count_2}/{total} samples evaded")
    
    sample_names = [s.rsplit('_', 1)[0] for s in model_evaded_samples]
    print("\n evaded samples: ", Counter(sample_names))      


 Random_forest: predictions

 No samples evaded for Random_forest

 Random_forest evasion rate: 0/174 samples evaded

 evaded samples:  Counter()

 XGB: predictions

 No samples evaded for XGB

 XGB evasion rate: 0/174 samples evaded

 evaded samples:  Counter()

 SVM_poly: predictions

 No samples evaded for SVM_poly

 SVM_poly evasion rate: 0/174 samples evaded

 evaded samples:  Counter()

 SVM_Linear: predictions

 SVM_Linear evasion rate: 122/174 samples evaded

 evaded samples:  Counter({'NominatusStorm': 11, 'None': 6, 'Azov': 5, 'FormAttachment.html': 4, 'Attachment.html': 4, 'RedLineStealer': 4, 'Conti': 4, 'UpdateRecords.html': 4, 'GandCrab': 4, 'DarkSide': 3, 'FORM.html': 3, 'UpdateInformation.html': 3, 'LegionLocker': 2, 'UpdateNow.html': 2, 'Attached.html': 2, 'Chaos': 2, 'UpdateForm.html': 2, 'FormAttached.html': 2, 'BlueSky': 2, 'WannaCry': 2, 'HiddenTear': 2, 'Update-Form110.html': 1, 'CryToy': 1, 'WastedLocker': 1, 'Update-Form143.html': 1, 'Ransomare.Pedo': 1, 'Downl